# Crypto tracker — a data pipeline you can read in one sitting

This notebook builds a small but complete analytics pipeline end to end, in Colab, with **zero local setup and no API keys**: raw prices and FX rates are extracted from two public APIs and loaded into a DuckDB warehouse with **dlt**, then transformed through a bronze → silver → gold medallion into a star schema with **dbt**.

It runs against the real [`crypto-tracker`](https://github.com/william-dwe/crypto-tracker) repository rather than a toy reimplementation: the notebook clones the repo and calls the same `ingest/` module and the same `transform/` dbt project that the scheduled pipeline runs. Everything lives under `/content` and disappears with the Colab VM.

**Shape of what follows.** Each half shows the machinery by hand first — one resource, one source, one pipeline, one model — and then hands off to the project's own entry point to do the full job. Reading the hand-built example is the point; the entry point is what you would actually run.

**Runtimes:** the install cell is 2–3 minutes (dbt-core is a large package). The first ingest is about 4 minutes — that is CoinGecko's keyless rate limit, not slow code. Everything else is seconds.

## Init & Setup
We going to install the same dlt + duckdb as the one we used on the `crypto-tracker` repo, besides we going to clone the entire repository as well, so that we can cross-referencing some pre-built functions from it.

You may found some different module import compared to the one on `crypto-tracker`, but all of them are necessary to ensure this code are executable through google collab. 

In [ ]:
# The same pins the repo's pyproject.toml uses. Pins are deliberate: dbt-core
# minor versions change macro/Jinja behaviour, and dlt's schema inference is
# version-sensitive too. Airflow is deliberately NOT installed — nothing here
# needs a scheduler, and installing with Airflow's constraints file would pin
# pathspec==1.1.1, which dbt-core cannot satisfy (it requires <1.1).
%pip install -q "dlt[duckdb]==1.30.0" "dbt-core==1.12.3" "dbt-duckdb==1.11.0" "duckdb==1.5.5"

# `--depth 1` keeps the clone small and fast; we only need HEAD.
!git clone --depth 1 https://github.com/william-dwe/crypto-tracker

# Same effect as the workshop's `export PYTHONPATH="$PWD"` — a visible one-liner
# so you can see how `from ingest import ...` resolves.
import sys
sys.path.insert(0, "/content/crypto-tracker")

# DB path must be the clone's own `data/crypto.duckdb`, not the default
# `ingest/__init__.py` (which would land in the wrong directory).
from ingest import DB_PATH, TRACKED_COINS, FIAT_CURRENCIES, DECIMAL_HINT, REPO_ROOT
print("REPO_ROOT:", REPO_ROOT)
print("DB_PATH:", DB_PATH)
print("TRACKED_COINS:", TRACKED_COINS)
print("FIAT_CURRENCIES:", FIAT_CURRENCIES)
print("DECIMAL_HINT:", DECIMAL_HINT)


## Concept recap

dlt's three moving parts and the one footgun you will hit in this module:

- **Resource** — a Python generator that yields rows (plain dicts). (reference: https://dlthub.com/docs/general-usage/resource)
- **Source** — a collection of related resources, run together. (reference: https://dlthub.com/docs/general-usage/source)
- **Pipeline** — binds a source to a destination (DuckDB here) and runs it, remembering state. (reference: https://dlthub.com/docs/general-usage/pipeline)

dlt inspects the **first batch of rows** and infers a column type automatically, so you won't need to define your schema manually. but, in real production ingestion, it always advised to have a pre-defined schema configuration that act as an api contract. To do that, you need to define the DLT schema object (reference: https://dlthub.com/docs/general-usage/schema).

# Ingesting CoinGecko API with dlt

Now we wire the three moving parts together for real, with the smallest possible slice of the pipeline: **one endpoint** (`/coins/markets`), **three coins**, **one run**.

The order below is the order dlt itself thinks in:

1. **resource** — a generator that yields plain dicts, plus the hints (`write_disposition`, `columns`) that tell dlt how to type and write them,
2. **source** — the group of resources that get loaded together and share one schema,
3. **pipeline** — the binding of that source to a destination (our DuckDB file) and the state that makes re-runs incremental,
4. **run** — extract → normalize → load, and the load package that records what happened.

This demo writes to its **own** pipeline name and its **own** schema (`bronze_demo`), so nothing here touches the real `bronze` tables that `run_ingest()` and dbt depend on. The full pipeline — both CoinGecko endpoints plus Frankfurter FX — is what `run_ingest()` does in the next section.

In [ ]:
# ---- 1. RESOURCE -------------------------------------------------------
# A resource is just a generator function that yields dicts, wrapped in
# @dlt.resource so dlt knows how to name, type and write the rows.
import dlt
import requests
from datetime import datetime, timezone
from decimal import Decimal

from ingest import COINGECKO_BASE_URL, DECIMAL_HINT

DEMO_COINS = ["bitcoin", "ethereum", "dogecoin"]
DEMO_NUMERIC_FIELDS = ["current_price", "market_cap", "total_volume"]


@dlt.resource(
    # `name` becomes the destination table name.
    name="coins_markets_demo",
    # A snapshot has no natural key we want to update: every run appends a new
    # generation of rows. The history resource in the repo uses "merge" with
    # primary_key=["coin_id", "price_ts_ms"] instead, so re-runs update in place.
    write_disposition="append",
    # DECIMAL_HINT = {"data_type": "decimal", "precision": 38, "scale": 18}.
    # Hints alone are NOT enough — see the Decimal() coercion below.
    columns={field: DECIMAL_HINT for field in DEMO_NUMERIC_FIELDS},
)
def coins_markets_demo(coins=None):
    # One timestamp for the whole run: it identifies the snapshot generation.
    # The API's own `last_updated` is per-coin and can be weeks stale.
    coins = coins if coins is not None else DEMO_COINS
    ingested_at = datetime.now(timezone.utc).isoformat()

    response = requests.get(
        f"{COINGECKO_BASE_URL}coins/markets",
        params={
            "vs_currency": "usd",
            "ids": ",".join(coins),
            "per_page": 250,
            "page": 1,
            "sparkline": False,
        },
        timeout=45,
    )
    response.raise_for_status()

    for row in response.json():
        out = {key: row.get(key) for key in ("id", "symbol", "name", *DEMO_NUMERIC_FIELDS)}
        for field in DEMO_NUMERIC_FIELDS:
            if out[field] is not None:
                out[field] = Decimal(str(out[field]))
        out["_ingested_at"] = ingested_at
        yield out


# Calling the decorated function returns a fresh, independently iterable
# resource — handy for eyeballing rows before any warehouse is involved.
preview = list(coins_markets_demo(coins=["bitcoin"]))
print(f"{len(preview)} row(s); first row:")
for key, value in preview[0].items():
    print(f"  {key:<16} {value!r}  ({type(value).__name__})")

In [ ]:
# ---- 2. SOURCE ---------------------------------------------------------
# A source groups resources that belong together, share one schema, and are
# loaded in one shot. Ours has a single resource; the repo's real pipeline
# runs three (markets snapshot, coin history, FX rates).
@dlt.source(name="coingecko_demo")
def coingecko_demo_source(coins=None):
    coins = coins if coins is not None else DEMO_COINS
    return coins_markets_demo(coins=coins)


demo_source = coingecko_demo_source()
print("resources in this source:", list(demo_source.resources.keys()))

# Same endpoint, different style: `ingest/coingecko.py::coingecko_markets_source`
# builds this source declaratively with dlt's REST API helper
# (`rest_api_source`) — client/paginator/params as config instead of a
# hand-written generator, with `apply_hints(...)` attaching the decimal hints.
# Both produce a source object; pick config when the API is well-behaved,
# a generator when you need custom pacing, retries, or row reshaping.

In [ ]:
# ---- 3. PIPELINE -------------------------------------------------------
# The pipeline binds a source to a destination and remembers state between
# runs (load ids, schema versions, incremental cursors).
#
# Two deliberate differences from the real pipeline in `ingest/run_ingest.py`:
#   * pipeline_name — dlt keeps per-pipeline state under ~/.dlt/pipelines/<name>.
#     Reusing "crypto_tracker" here would let this demo scribble on the state
#     that run_ingest() owns.
#   * dataset_name  — the real run lands in `bronze`, which dbt's `br_*` trust
#     views read. The demo gets its own `bronze_demo` schema in the SAME
#     DuckDB file, so it is visible but harmless.
DEMO_PIPELINE_NAME = "crypto_tracker_nb_demo"
DEMO_DATASET = "bronze_demo"

demo_pipeline = dlt.pipeline(
    pipeline_name=DEMO_PIPELINE_NAME,
    destination=dlt.destinations.duckdb(DB_PATH),
    dataset_name=DEMO_DATASET,
)
print("pipeline:", demo_pipeline.pipeline_name)
print("dataset :", demo_pipeline.dataset_name)
print("file    :", DB_PATH)

In [ ]:
# ---- 4. RUN ------------------------------------------------------------
# extract (call the API) -> normalize (infer/apply schema, flatten) ->
# load (write to DuckDB). Re-running this cell appends another snapshot,
# because the resource's write_disposition is "append".
load_info = demo_pipeline.run(demo_source)
print(load_info)
print()
print(demo_pipeline.last_trace.last_normalize_info)

# What did dlt decide the columns are? This is the proof that the decimal
# hint + Decimal() coercion worked: no `*__v_double` twin anywhere.
print()
print("inferred schema for coins_markets_demo:")
columns = demo_pipeline.default_schema.get_table("coins_markets_demo")["columns"]
for name, column in columns.items():
    print(f"  {name:<20} {column.get('data_type')}")

# And the rows themselves. Read-only connection, opened and closed inside the
# same cell: DuckDB is single-writer, so never leave one open across cells.
import duckdb

con = duckdb.connect(DB_PATH, read_only=True)
try:
    rows = con.execute(
        f"select id, symbol, current_price, market_cap, _ingested_at "
        f"from {DEMO_DATASET}.coins_markets_demo order by market_cap desc"
    ).fetchall()
    for row in rows:
        print(" ", row)
finally:
    con.close()

We can have the same result by using the `run_ingest()` function that has been defined on the `crypto-tracker` repo. Behind the scene it also defines `resource`, `source`, and `pipeline` as well.

The difference is scope: the demo above loaded three coins from one endpoint into its own `bronze_demo` schema, while `run_ingest()` loads all ten tracked coins from both CoinGecko endpoints plus Frankfurter's FX rates into the real `bronze` schema — the landing zone the dbt half transforms. Expect about 4 minutes: CoinGecko's keyless tier is paced at 3 seconds between per-coin history requests.


In [ ]:
from ingest.run_ingest import run_ingest

run_ingest()

In [ ]:
# Inspect the bronze layer we just landed.
#
# DuckDB is single-writer: opening a connection while another process holds
# the write lock fails. We open read-only, do the work, and close in the
# same cell — never let a connection span cell boundaries in a notebook.
import duckdb

con = duckdb.connect(DB_PATH, read_only=True)
try:
    # Every bronze table dlt owns, ordered alphabetically for stable output.
    tables = con.execute(
        "select schema_name, table_name, estimated_size "
        "from duckdb_tables() where schema_name='bronze' order by table_name"
    ).fetchall()
    print(f"{'schema':<8} {'table':<28} {'rows (est.)':>12}")
    for schema, name, size in tables:
        print(f"{schema:<8} {name:<28} {size:>12,}")

    print()
    # Source-table row counts, exact this time.
    for t in ("coins_markets_raw", "coin_market_chart_raw", "fx_rates_raw"):
        n = con.execute(f"select count(*) from bronze.{t}").fetchone()[0]
        print(f"bronze.{t:<26} {n:>10,} rows")

    print()
    # dlt's own bookkeeping: one row per successful load. status=0 means
    # clean; the dbt `br_completed_loads` view filters on exactly that.
    print("dlt load log (most recent 5):")
    rows = con.execute(
        "select load_id, schema_version_hash, status, inserted_at "
        "from bronze._dlt_loads order by inserted_at desc limit 5"
    ).fetchall()
    for load_id, svh, status, ts in rows:
        print(f"  {load_id}  status={status}  at={ts}  svh={svh[:12]}...")

    print()
    # Peek at a snapshot row. Notice `current_price` and friends land as
    # DECIMAL(38,18) — that's DECIMAL_HINT + _coerce_numerics working together.
    print("Sample of bronze.coins_markets_raw:")
    sample = con.execute(
        "select id, symbol, name, current_price, market_cap "
        "from bronze.coins_markets_raw order by market_cap desc nulls last limit 5"
    ).fetchall()
    for r in sample:
        print(" ", r)
finally:
    con.close()

# Transforming the warehouse with dbt

The bronze schema now holds exactly what the APIs returned, with dlt's bookkeeping alongside it. That is the raw landing zone; nothing in it is safe to hand to an analyst yet. dbt is what turns it into a star schema.

dbt is SQL with a dependency graph. You write `select` statements; dbt works out the order and builds them.

- **`source()`** — read a raw table dbt does not own (the dlt landing zone). Sources are leaves: no DAG edge points into them.
- **`ref()`** — read another model *and* wire the dependency edge. `gold.dim_coin` builds after `silver.stg_coin_snapshots` because its SQL says `ref('stg_coin_snapshots')`.
- **Materialization** — how a model lands: `view` (computed on read, no disk) or `table`. `transform/dbt_project.yml` sets this per folder: bronze and silver are views, gold is tables.

Two traps this project has already solved for you, both worth knowing before you run anything:

- **The trust filter.** dlt writes rows *before* it marks a load successful, so an interrupted run can leave rows behind whose load id never reaches `status = 0`. Selecting straight from a raw table would silently include them. The bronze models inner-join a completed-loads view so those rows never reach gold.
- **The `main_` prefix.** dbt-duckdb builds custom schema names as `<target-schema>_<custom>` by default, so `+schema: gold` would land in `main_gold`. `transform/macros/generate_schema_name.sql` overrides `generate_schema_name` to use the custom name verbatim.


In [ ]:
# Read the two models the trust filter is made of, straight from the clone.
# Printing the real files (rather than pasting copies here) means this cell
# can never drift from what actually runs.
from pathlib import Path

from IPython.display import Markdown, display

TRANSFORM = Path("/content/crypto-tracker/transform")

for rel in ("models/bronze/br_completed_loads.sql", "models/bronze/br_coins_markets.sql"):
    body = (TRANSFORM / rel).read_text()
    display(Markdown(f"**`transform/{rel}`**\n\n```sql\n{body}\n```"))


Before building all 18 models, run exactly one of them. `br_completed_loads` is the right one to start with: it is the leaf the whole trust boundary hangs off, it depends on nothing but the dlt bookkeeping table, and it is a view, so building it is instant.

Both dbt flags below are ones people forget:

- **`CRYPTO_DB_PATH`** — `transform/profiles.yml` reads `{{ env_var('CRYPTO_DB_PATH') }}` and has no default, so dbt errors out without it.
- **`--profiles-dir .`** — the profile lives in the project directory, not in the default `~/.dbt/`.


In [ ]:
# Every dbt command runs from the project root, with the two flags above.
# %cd and %env are notebook-level so the `!dbt` subprocesses below inherit them.
%cd /content/crypto-tracker/transform
%env CRYPTO_DB_PATH=/content/crypto-tracker/data/crypto.duckdb

# Build one model, then show what it returns. --select takes a model name;
# `dbt show` compiles the model and prints rows without materialising anything.
!dbt run --select br_completed_loads --profiles-dir .
!dbt show --select br_completed_loads --limit 5 --profiles-dir .


That was one model. `dbt build` is the entry point that does the rest: it runs every model in dependency order, loads the seed, and runs all 80 data tests — the same command the scheduled pipeline runs.

One flag is needed on this first build. `gold.fct_coin_price_daily` is an incremental model: on later runs it reprocesses only a recent window and reaches back 7 days so its `lag()` window still sees preceding rows. On a warehouse where the table does not exist yet there is nothing to reach back into, so the first build has to be a full refresh. After this, plain `dbt build` is the right command.


In [ ]:
# The full medallion: 18 models, 1 seed, 80 tests, in dependency order.
# --full-refresh because gold.fct_coin_price_daily is incremental and this is
# its first build; every subsequent build can drop the flag.
!dbt build --full-refresh --profiles-dir .


In [ ]:
# Inspect the transformed warehouse. Read-only connection, opened and closed in
# the same cell: DuckDB is single-writer, so never let one span cell boundaries.
import duckdb

DB = "/content/crypto-tracker/data/crypto.duckdb"

con = duckdb.connect(DB, read_only=True)
try:
    # Note the schemas are bronze / silver / gold, NOT main_bronze / main_silver
    # / main_gold — that is generate_schema_name doing its job.
    print(f"{'schema':<8} {'name':<28} {'type':<12}")
    for schema, name, kind in con.execute(
        """
        select table_schema, table_name, table_type
        from information_schema.tables
        where table_schema in ('bronze', 'silver', 'gold')
        order by table_schema, table_name
        """
    ).fetchall():
        print(f"{schema:<8} {name:<28} {kind:<12}")

    print()
    print("gold.dim_coin:")
    for r in con.execute(
        "select coin_id, symbol, days_of_history, is_active "
        "from gold.dim_coin order by coin_id"
    ).fetchall():
        print(" ", r)

    print()
    print("gold.fct_coin_price_daily — most recent 5 rows:")
    for r in con.execute(
        "select coin_id, price_date, price_usd, daily_return_pct "
        "from gold.fct_coin_price_daily order by price_date desc, coin_id limit 5"
    ).fetchall():
        print(" ", r)
finally:
    con.close()


## Cleanup + next steps

**Colab's runtime is ephemeral.** Everything this notebook made lives under `/content` — the clone and its `data/crypto.duckdb` — and is gone when the VM shuts down (12 h idle timeout, 90 min after the last cell in the free tier). To keep the warehouse, download it from the file browser in the left sidebar, or:

```python
from google.colab import files
files.download("/content/crypto-tracker/data/crypto.duckdb")
```

**What this notebook did not cover:** orchestration. The repo runs both halves on a daily Airflow DAG (`dags/crypto_tracker_daily.py`, two tasks, `ingest_raw >> dbt_build`), where a one-slot `duckdb_writer` pool serialises write access — the same single-writer constraint you saw here as "close the connection in the same cell". Airflow needs a scheduler and a metadata database, so it belongs on a real machine rather than in a notebook.

**Running the real thing locally:**

```bash
git clone https://github.com/william-dwe/crypto-tracker
cd crypto-tracker
uv sync                       # one command, one venv, all pins
uv run ct airflow-init        # one-time: migrate Airflow DB + create the duckdb_writer pool
uv run ct run                 # ingest + dbt build
uv run ct ui                  # portfolio + performance on the gold layer
```

Full walkthrough: [`docs/workshop.md`](https://github.com/william-dwe/crypto-tracker/blob/main/docs/workshop.md). Architecture rationale: [`docs/ARCHITECTURE.md`](https://github.com/william-dwe/crypto-tracker/blob/main/docs/ARCHITECTURE.md).
